# Notebook 03: Privacy Analysis  

This notebook performs a privacy and governance analysis of the raw nested JSON dataset (`raw_credit_applications.json`).

The objective is to:

 - Identify personally identifiable information (PII) contained in the dataset

 - Detect fields that may pose privacy or re-identification risks

 - Apply pseudonymization techniques to sensitive identifiers

 - Demonstrate privacy-preserving practices such as data minimization

 - Highlight potential data protection and governance risks related to the use of the dataset in automated decision-making systems

This notebook is structured as follows:

1. **Dataset Structure Overview**
   - Load and inspect the raw JSON dataset
   - Flatten the nested structure to facilitate analysis

2. **PII Identification**
   - Detect fields containing direct identifiers (e.g., SSN, email, full name)
   - Identify attributes that may increase the risk of re-identification

3. **Pseudonymization of Sensitive Fields**
   - Apply hashing techniques to selected identifiers
   - Remove raw identifiers to enforce data minimization

4. **Privacy Risk Assessment**
   - Evaluate privacy risks with reference to **GDPR principles**
   - Discuss risks related to the processing of personal data

5. **Governance Considerations**
   - Highlight governance implications for the use of the dataset in **credit scoring systems**
   - Discuss regulatory aspects, including the classification of credit scoring as a **high-risk AI system under the EU AI Act**

6. **GDPR Compliance and Privacy Mitigation Measures**
   - Identify privacy risks present in the raw dataset in relation to key **GDPR principles**
   - Explain the mitigation measures applied, including **pseudonymization and data minimization**, to reduce the risk of re-identification and improve compliance with data protection requirements

# Part 1: Dataset Structure Overview

In this section, the raw JSON dataset is loaded and inspected to understand its structure. The nested fields are flattened into a tabular format to facilitate further analysis.  This step allows us to explore the dataset dimensions, examine the available variables, and identify the main attributes that will be used in the subsequent privacy and governance analysis.

In [10]:
import json
import pandas as pd

# Load the JSON dataset from the data folder
with open("../data/raw_credit_applications.json", "r") as f:
    data = json.load(f)

# Flatten nested JSON structure
df = pd.json_normalize(data)

print("Dataset loaded")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.head()

Dataset loaded
Rows: 502
Columns: ['_id', 'spending_behavior', 'processing_timestamp', 'applicant_info.full_name', 'applicant_info.email', 'applicant_info.ssn', 'applicant_info.ip_address', 'applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code', 'financials.annual_income', 'financials.credit_history_months', 'financials.debt_to_income', 'financials.savings_balance', 'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate', 'decision.approved_amount', 'financials.annual_salary', 'notes']


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


# Part 2: PII Identification

In this section, the dataset is analyzed to identify fields that may contain **personally identifiable information (PII)** or attributes that could contribute to the re-identification of individuals.  The classification focuses on detecting **direct identifiers** (e.g., SSN, email, full name) as well as other attributes that may increase privacy risks when combined with additional information.

In [11]:
# Simple PII discovery based on column names
keywords = ["ssn", "email", "full_name", "ip", "date_of_birth", "dob", "zip", "gender"]
pii_cols = [c for c in df.columns if any(k in c.lower() for k in keywords)]

print("Potential PII / quasi-PII columns detected:")
for c in pii_cols:
    print("-", c)

Potential PII / quasi-PII columns detected:
- applicant_info.full_name
- applicant_info.email
- applicant_info.ssn
- applicant_info.ip_address
- applicant_info.gender
- applicant_info.date_of_birth
- applicant_info.zip_code


# Part 3: Pseudonymization of Sensitive Fields

In this section, selected fields containing direct identifiers are pseudonymized using a hashing function. This process replaces sensitive values with anonymized representations while preserving the structure of the dataset. The goal is to reduce the risk of re-identification and demonstrate the application of privacy-preserving techniques such as data minimization and pseudonymization before using the data for analysis.

In [12]:
import hashlib
import os

# Secret pepper used to strengthen the hash
PEPPER = os.environ.get("NOVACRED_PEPPER", "ORION_GOV_PEPPER_2026!x7")

# Pseudonymization function (deterministic hashing)
def pseudo(x):
    if pd.isna(x) or x == "":
        return x
    value = (str(x).strip() + PEPPER).encode("utf-8")
    return hashlib.sha256(value).hexdigest()

# Select the main direct identifiers to pseudonymize (if present)
cols_to_hash = ["applicant_info.ssn", "applicant_info.email", "applicant_info.full_name"]

for c in cols_to_hash:
    if c in df.columns:
        df[c + "_hashed"] = df[c].apply(pseudo)

# Remove the original raw PII columns (data minimization)
df_priv = df.drop(columns=[c for c in cols_to_hash if c in df.columns])

print("Pseudonymization completed")
df_priv[["_id"] + [c + "_hashed" for c in cols_to_hash if c + "_hashed" in df_priv.columns]].head()

Pseudonymization completed


,_id,applicant_info.ssn_hashed,applicant_info.email_hashed,applicant_info.full_name_hashed
0,app_200,abeaf4b4df44354d7e894fd38c4595bf2774bbec2c6df8...,698228d9c2d921ce0ac3040a558c7c71bd862567da0732...,11aae2e2e0016ea812800d3fc81aa700be324c6c7bb251...
1,app_037,9f0a81c00bc0f56da8b58b2a9d026bb3eed50239640358...,be8ea5d490565a5b2bba34ae986b7a73b0356ab380c6fb...,8d4d6f16b818ce789aa8dae31db0b42723a29bac5e966f...
2,app_215,552dd4f85965458f472dbc02a09e1ff2884fcc78bca5f8...,85e4d75f3a6c021d787ecdeae4a5a5d7f4c44dc661f27a...,848dd37299065284d355e1e126b28199ad6d1a02a6cbb5...
3,app_024,6aa71343c7411cebd3d841ec098f401ccd3ef2673eceb3...,01e48de31d87882ec2b2cb2fdcc94b2d9fc1903fa90345...,a1bb25035971911454b10a3c3deea276ecacd0ca77c174...
4,app_184,db89677a88e18acef39b9af65519cec3809942b06ad560...,e2a63f5ea58d3a09ea51a58a6f50d65b4ce89ab5d00a84...,c71a217d593ef2fb521d800e620f6b8043faed0efb3a64...


# Part 4: Privacy Risk Assessment

In this section, the dataset is examined to identify potential privacy risks related to the processing of personal data. The analysis highlights how the presence of direct identifiers and demographic attributes may increase the risk of re-identification or misuse. These observations are discussed in relation to key **GDPR principles**, such as data minimization, purpose limitation, and data protection.

In [13]:
# Quick privacy risk indicators (simple descriptive checks)

# Count missing values in detected PII/quasi-PII columns
missing_summary = df[pii_cols].isna().mean().sort_values(ascending=False) if pii_cols else None

print("Missing rate (share of nulls) in detected PII/quasi-PII columns:")
if missing_summary is not None:
    print(missing_summary)
else:
    print("No PII/quasi-PII columns detected.")

Missing rate (share of nulls) in detected PII/quasi-PII columns:
applicant_info.ssn              0.009960
applicant_info.ip_address       0.009960
applicant_info.gender           0.001992
applicant_info.date_of_birth    0.001992
applicant_info.zip_code         0.001992
applicant_info.email            0.000000
applicant_info.full_name        0.000000
dtype: float64


# Part 5: Governance Considerations

In this section, the main governance implications of using the dataset are highlighted. The analysis considers how the presence of personal and demographic data may impact compliance with data protection regulations and responsible AI practices. Particular attention is given to the regulatory context of **credit scoring systems**, which are classified as **high-risk AI systems under the EU AI Act**, and therefore require stronger governance, transparency, and risk management measures.

In [14]:
# Simple governance-oriented summary to support documentation

direct_identifiers = [c for c in ["applicant_info.ssn", "applicant_info.email", "applicant_info.full_name", "applicant_info.ip_address"] if c in df.columns]
sensitive_or_proxy = [c for c in ["applicant_info.gender", "applicant_info.date_of_birth", "applicant_info.zip_code"] if c in df.columns]

print("Governance summary:")
print("- Direct identifiers found:", direct_identifiers)
print("- Sensitive/proxy attributes found:", sensitive_or_proxy)
print("- Recommended action: pseudonymize direct identifiers and remove originals (data minimization).")
print("- Note: Credit scoring is a high-risk use case under the EU AI Act, requiring strong governance controls.")

Governance summary:
- Direct identifiers found: ['applicant_info.ssn', 'applicant_info.email', 'applicant_info.full_name', 'applicant_info.ip_address']
- Sensitive/proxy attributes found: ['applicant_info.gender', 'applicant_info.date_of_birth', 'applicant_info.zip_code']
- Recommended action: pseudonymize direct identifiers and remove originals (data minimization).
- Note: Credit scoring is a high-risk use case under the EU AI Act, requiring strong governance controls.


# Part 6: GDPR Compliance and Privacy Mitigation Measures